# Mufahras: search by meaning, round 2 (what to embed)

Round 1 (`meaning-spike.ipynb`) picked the model family. This round keeps the three best small models and tests
**what text to embed**, each change on its own and then together:

| Variant | Verses | Hadith |
|---|---|---|
| `base` | translation | full English (round 1) |
| `verse+tafsir` | translation followed by its Al-Mukhtasar explanation, one vector | as `base` |
| `verse|tafsir` | two vectors, translation and explanation; a verse scores as the better one | as `base` |
| `no-opening` | as `base` | without the opening ("Narrated Anas:", "Abu Huraira reported … as saying:") |
| `passages` | as `base` | without the opening, long hadith split into overlapping passages; scores as its best passage |
| `all` | best verse option | `passages` |

It also scores against the **accepted alternatives** added to `eval/meaning-queries.json` (the `also` lists:
correct texts the round 1 models found, often the same hadith in another book), and tries weighted blends with
keyword search.

Nothing to upload. A GPU run takes about 10–15 minutes. At the end, save `results-2.csv`, `misses-2.csv` and
`rankings-2.json` into `eval/` and tell me.

In [ ]:
%pip install -q -U sentence-transformers pandas
import torch; print(torch.__version__, torch.cuda.is_available() and torch.cuda.get_device_name(0))

## The texts and the queries

In [ ]:
import json, re, urllib.request, numpy as np, pandas as pd, time, os
BASE = 'https://omeiirr.github.io/mufahras/data/'
REPO = 'https://raw.githubusercontent.com/omeiirr/mufahras/main/'
get = lambda url: json.load(urllib.request.urlopen(url))
manifest = get(BASE + 'manifest.json')['chunks']
chunk = lambda name: get(BASE + manifest[name]['file'])
clean = lambda t: re.sub(r'\s+', ' ', re.sub('[\u00B9\u00B2\u00B3\u2070-\u209F]', '', t)).strip()

quran, tafsir = chunk('quran'), chunk('tafsir')
verse_refs, verses, explanations = [], [], []
for s in quran['surahs']:
    for a in range(1, s['count'] + 1):
        i = s['start'] + a - 1
        t = tafsir['text'][i]
        if isinstance(t, int): t = tafsir['text'][t]   # verses in a group share the group's explanation
        verse_refs.append(f"{s['n']}:{a}"); verses.append(clean(quran['en'][i])); explanations.append(clean(t or ''))

hadith_refs, hadith = [], []
for book in ['bukhari', 'muslim', 'abudawud', 'tirmidhi', 'nasai', 'ibnmajah']:
    for h in chunk(f'hadith-{book}')['hadiths']:
        if h[2] and h[2].strip():
            hadith_refs.append(f'{book} {h[0]}'); hadith.append(clean(h[2]))

# The opening names the narrators; it carries no meaning a search would match.
OPENING = re.compile(r'^[^:]{0,200}?\b(narrat\w*|report\w*|saying|said|authority|transmitted)\b[^:]{0,80}?:\s*', re.I)
SITUATION = re.compile(r'\b(asked|question|came to|went to|said to|told|when|while|whether|decided|saw|heard that)\b', re.I)
def strip_opening(t):
    m = OPENING.match(t)
    # Longer "openings" include the question or the situation, which is meaning: keep those.
    if not m or m.end() > 140 or len(t) - m.end() < 20 or SITUATION.search(t[:m.end()]): return t
    return t[m.end():]
hadith_bare = [strip_opening(t) for t in hadith]

# Long hadith: windows of 180 words, every 120 words, so a passage fits the 256-token window.
def passages(t, size=180, step=120):
    w = t.split()
    if len(w) <= size + 40: return [t]
    return [' '.join(w[i:i + size]) for i in range(0, max(1, len(w) - size + step), step) if w[i:i + size]]
passage_texts, passage_owner = [], []
for j, t in enumerate(hadith_bare):
    for p in passages(t): passage_texts.append(p); passage_owner.append(j)
passage_owner = np.array(passage_owner)

queries = get(REPO + 'eval/meaning-queries.json')
keyword = get(REPO + 'eval/keyword-baseline.json')
refs = verse_refs + hadith_refs
print(len(verses), 'verses,', len(hadith), 'hadith,', len(passage_texts), 'hadith passages,', len(queries), 'queries,',
      sum(len(q.get('also', {})) for q in queries), 'accepted alternatives')
for t in [hadith[0], hadith[8000], hadith[20000], hadith[30000]]:   # one from several books
    print('  ', t[:90], '\n →', strip_opening(t)[:90])

## Models

In [ ]:
BGE_Q = 'Represent this sentence for searching relevant passages: '
MODELS = {
    'bge-small': ('BAAI/bge-small-en-v1.5', BGE_Q, ''),
    'e5-small':  ('intfloat/e5-small-v2', 'query: ', 'passage: '),
    'arctic-s':  ('Snowflake/snowflake-arctic-embed-s', BGE_Q, ''),
}
MAX_TOKENS = 256

## Scoring

In [ ]:
K = 10
def score(ranked):
    """hit@10 and MRR count the original answers or an accepted alternative; recall@10 only the original ones."""
    strict = hit = rec = mrr = 0; kinds = {}
    for q in queries:
        exp, ok = list(q['expect']), set(q['expect']) | set(q.get('also', {}))
        top = ranked[q['query']][:K]
        found = sum(r in top for r in exp)
        first = next((i for i, r in enumerate(top) if r in ok), None)
        strict += found > 0; hit += first is not None; rec += found / len(exp)
        mrr += 0 if first is None else 1 / (first + 1)
        k = kinds.setdefault(q['kind'], [0, 0]); k[0] += first is not None; k[1] += 1
    n = len(queries)
    return {'hit@10': round(100 * hit / n), 'strict hit@10': round(100 * strict / n), 'recall@10': round(100 * rec / n),
            'MRR': round(mrr / n, 2), **{kind: f'{h}/{t}' for kind, (h, t) in kinds.items()}}

def rrf(a, b, w=1.0):
    """Reciprocal-rank fusion; w weights the second ranking."""
    s = {}
    for lst, wt in ((a, 1.0), (b, w)):
        for i, r in enumerate(lst): s[r] = s.get(r, 0) + wt / (60 + i)
    return sorted(s, key=s.get, reverse=True)

def ranking(verse_scores, hadith_scores, n=50):
    """Per query: refs of the n best texts across verses and hadith."""
    allv = np.concatenate([verse_scores, hadith_scores], axis=1)
    top = np.argsort(-allv, axis=1)[:, :n]
    return {q['query']: [refs[i] for i in top[j]] for j, q in enumerate(queries)}

def best_passage(passage_scores):
    """(queries × passages) → (queries × hadith): each hadith scores as its best passage."""
    out = np.full((passage_scores.shape[0], len(hadith)), -1.0, dtype=np.float32)
    for j in range(passage_scores.shape[0]):
        np.maximum.at(out[j], passage_owner, passage_scores[j])
    return out

rows = [{'model': 'keyword (today)', 'variant': '', **score(keyword)}]
rankings = {'keyword': keyword}

## Run

In [ ]:
from sentence_transformers import SentenceTransformer
for key, (mid, qp, dp) in MODELS.items():
    model = SentenceTransformer(mid, device='cuda'); model.max_seq_length = MAX_TOKENS
    enc = lambda xs, p=dp: model.encode([p + x for x in xs], batch_size=256, normalize_embeddings=True,
                                        convert_to_numpy=True, show_progress_bar=False)
    t = time.time()
    E = {'verse': enc(verses), 'verse+tafsir': enc([v + ' ' + e for v, e in zip(verses, explanations)]),
         'tafsir': enc(explanations), 'hadith': enc(hadith), 'hadith-bare': enc(hadith_bare),
         'passages': enc(passage_texts)}
    Q = enc([q['query'] for q in queries], qp)
    print(f'{key}: embedded in {time.time() - t:.0f}s')
    S = {name: Q @ v.T for name, v in E.items()}
    v_dual = np.maximum(S['verse'], S['tafsir'])
    h_pass = best_passage(S['passages'])
    variants = {
        'base': (S['verse'], S['hadith']),
        'verse+tafsir': (S['verse+tafsir'], S['hadith']),
        'verse|tafsir': (v_dual, S['hadith']),
        'no-opening': (S['verse'], S['hadith-bare']),
        'passages': (S['verse'], h_pass),
        'all (verse+tafsir)': (S['verse+tafsir'], h_pass),
        'all (verse|tafsir)': (v_dual, h_pass),
    }
    for name, (vs, hs) in variants.items():
        r = ranking(vs, hs); rankings[f'{key} / {name}'] = r
        rows.append({'model': key, 'variant': name, **score(r)})
    for name in ('all (verse+tafsir)', 'all (verse|tafsir)'):
        sem = rankings[f'{key} / {name}']
        for w in (0.5, 1.0, 2.0):
            r = {q: rrf(keyword[q], sem[q], w) for q in sem}
            rankings[f'{key} / {name} + keyword w{w}'] = r
            rows.append({'model': key, 'variant': f'{name} + keyword (semantic ×{w})', **score(r)})
    del model; torch.cuda.empty_cache()

results = pd.DataFrame(rows).set_index(['model', 'variant'])
pd.set_option('display.max_rows', 200); pd.set_option('display.width', 250)
results

## Where each configuration misses

Rank (1–10) of the first accepted text, or `-`. One column per model with its `all` variants and the keyword baseline.

In [ ]:
cols = ['keyword'] + [k for k in rankings if k.endswith('all (verse+tafsir)') or k.endswith('all (verse|tafsir)')]
miss = pd.DataFrame({c: [next((str(i + 1) for i, r in enumerate(rankings[c][q['query']][:K])
                               if r in q['expect'] or r in q.get('also', {})), '-') for q in queries] for c in cols},
                    index=[f"{q['kind']}: {q['query']}" for q in queries])
pd.set_option('display.max_colwidth', 70)
miss

## Look at one query

In [ ]:
text_of = dict(zip(refs, verses + hadith))
def show(query, config='bge-small / all (verse|tafsir)', n=10):
    q = next(x for x in queries if x['query'] == query)
    for r in rankings[config][query][:n]:
        mark = '*' if r in q['expect'] else '+' if r in q.get('also', {}) else ' '
        print(mark, r.ljust(14), text_of[r][:110])
    print('expected:', ', '.join(q['expect']), '| also:', ', '.join(q.get('also', {})) or '-')
show('actions are judged by their intentions')

## Save

Save these three files into the repo's `eval/` folder.

In [ ]:
results.to_csv('results-2.csv'); miss.to_csv('misses-2.csv')
json.dump(rankings, open('rankings-2.json', 'w'))
!ls -lh results-2.csv misses-2.csv rankings-2.json